In [1]:
# =============================================================================
# 01_TURKEY_ECOREGIONS.ipynb
# Fire Season Timing — Turkey Ecoregion Scale
# =============================================================================
# Computes fire season timing metrics (onset, peak, end, season length)
# for all WWF RESOLVE ecoregions intersecting Turkey, for years 2003–2024.
#
# Data sources:
#   - MODIS Terra active fire: MODIS/061/MOD14A1
#   - MODIS Aqua active fire:  MODIS/061/MYD14A1
#   - Ecoregions:              RESOLVE/ECOREGIONS/2017
#   - Country boundary:        USDOS/LSIB_SIMPLE/2017
#
# Output:
#   - outputs/turkey_ecoregions/<ECO_ID>_<ECO_NAME>.csv  (per ecoregion)
#   - outputs/turkey_ecoregions/master_turkey.csv        (combined)
# =============================================================================

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import os
import time

# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='fire-seasons')

In [2]:
# =============================================================================
# PARAMETERS
# =============================================================================

FIRE_MASK_MIN   = 8     # FireMask threshold: >= 8 = nominal + high confidence only
ONSET_THRESHOLD = 0.05  # Cumulative fraction threshold for fire season onset (5%)
END_THRESHOLD   = 0.95  # Cumulative fraction threshold for fire season end (95%)
MIN_DETECTIONS  = 20    # Minimum annual fire detections required to compute metrics
YEARS           = list(range(2003, 2025))  # Full study period: 2003–2024

In [3]:
# =============================================================================
# LOAD MODIS COLLECTIONS
# =============================================================================
# Terra and Aqua are loaded once here at module level.
# Per-year and per-day filtering is handled inside get_daily_counts().

terra = ee.ImageCollection("MODIS/061/MOD14A1").select('FireMask')
aqua  = ee.ImageCollection("MODIS/061/MYD14A1").select('FireMask')

print('Terra and Aqua collections loaded.')

Terra and Aqua collections loaded.


In [4]:
# =============================================================================
# LOAD TURKEY ECOREGIONS
# =============================================================================

# Turkey boundary from LSIB
turkey = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
           .filter(ee.Filter.eq('country_na', 'Turkey'))

# RESOLVE ecoregions clipped to Turkey
ecoregions_turkey = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017") \
                      .filterBounds(turkey.geometry())

# Inspect
n_eco    = ecoregions_turkey.size().getInfo()
eco_list = ecoregions_turkey.select(['ECO_ID', 'ECO_NAME']).getInfo()

print(f'Number of ecoregions intersecting Turkey: {n_eco}')
print()
for f in eco_list['features']:
    print(f['properties']['ECO_ID'], '|', f['properties']['ECO_NAME'])

Number of ecoregions intersecting Turkey: 14

786 | Anatolian conifer and deciduous mixed forests
804 | Southern Anatolian montane conifer and deciduous forests
646 | Balkan mixed forests
650 | Caucasus mixed forests
665 | Euxine-Colchic broadleaf forests
652 | Central Anatolian steppe and woodlands
662 | Eastern Anatolian deciduous forests
688 | Zagros Mountains forest steppe
703 | Northern Anatolian conifer and deciduous forests
725 | Central Anatolian steppe
727 | Eastern Anatolian montane steppe
739 | Syrian xeric grasslands and shrublands
785 | Aegean and Western Turkey sclerophyllous and mixed forests
791 | Eastern Mediterranean conifer-broadleaf forests


In [5]:
# =============================================================================
# BUILD ECOREGION RECORD LIST
# =============================================================================
# Converts the GEE FeatureCollection into a plain Python list of dicts
# so we can iterate over ecoregions without repeated GEE calls.

eco_records = []
for f in eco_list['features']:
    eco_records.append({
        'eco_id'   : f['properties']['ECO_ID'],
        'eco_name' : f['properties']['ECO_NAME'],
        'geometry' : ee.Geometry(f['geometry'])
    })

print(f'Loaded {len(eco_records)} ecoregion records.')

Loaded 14 ecoregion records.


In [6]:
# =============================================================================
# FUNCTION: get_daily_counts
# =============================================================================

def get_daily_counts(eco_geometry, year):
    """
    Compute daily MODIS active fire detection counts for a given
    ecoregion geometry and calendar year.

    Combines Terra (MOD14A1) and Aqua (MYD14A1) by taking the pixel-wise
    maximum across sensors for each day, which deduplicates detections
    that appear in both sensors on the same day.

    Only pixels with FireMask >= FIRE_MASK_MIN (nominal + high confidence)
    are counted as fire detections.

    All daily reductions are computed server-side via a mapped
    ee.List, and results are fetched in two getInfo() calls
    (one for DOYs, one for counts), minimising client-server
    round trips.

    Parameters
    ----------
    eco_geometry : ee.Geometry
        The geometry of the ecoregion to compute counts for.
    year : int
        The calendar year to process (e.g. 2008).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
          - doy           : int, day of year (1-indexed)
          - n_detections  : int, number of fire pixels detected
        One row per day of the year (365 or 366 rows).
    """

    import calendar

    start  = ee.Date.fromYMD(year, 1, 1)
    end    = ee.Date.fromYMD(year + 1, 1, 1)
    n_days = 366 if calendar.isleap(year) else 365

    # Pre-filter both collections to this year — reduces per-day filter overhead
    terra_year = terra.filterDate(start, end)
    aqua_year  = aqua.filterDate(start, end)

    # Fallback empty image for days where a sensor returns no image
    empty = ee.Image.constant(0).rename('FireMask').toUint8()

    # Server-side sequence of day offsets: 0, 1, 2, ..., n_days-1
    day_seq = ee.List.sequence(0, n_days - 1)

    def make_daily_image(d):
        """
        For a single day offset d, build a deduplicated binary fire image
        and reduce it to a pixel count stored as an image property.
        Returns a 1-pixel constant image carrying 'doy' and 'count' properties.
        """
        d        = ee.Number(d)
        date     = start.advance(d, 'day')
        date_end = date.advance(1, 'day')

        # Filter each sensor to this single day; fall back to empty if no image
        terra_day = terra_year.filterDate(date, date_end)
        aqua_day  = aqua_year.filterDate(date, date_end)

        t = ee.Image(ee.Algorithms.If(
            terra_day.size().gt(0),
            terra_day.select('FireMask').max(),
            empty
        ))
        a = ee.Image(ee.Algorithms.If(
            aqua_day.size().gt(0),
            aqua_day.select('FireMask').max(),
            empty
        ))

        # Pixel-wise max across sensors = deduplication
        combined    = t.max(a)
        fire_binary = combined.gte(FIRE_MASK_MIN).unmask(0).rename('fire')

        # Reduce to a single count value and store as image property
        count = fire_binary.reduceRegion(
            reducer   = ee.Reducer.sum(),
            geometry  = eco_geometry,
            scale     = 1000,
            maxPixels = 1e8
        ).get('fire')

        return ee.Image.constant(0).set('doy', d.add(1)).set('count', count)

    # Map over all days server-side — no Python loop
    daily_collection = ee.ImageCollection(day_seq.map(make_daily_image))

    # Fetch DOYs and counts in two getInfo() calls
    doys   = daily_collection.aggregate_array('doy').getInfo()
    counts = daily_collection.aggregate_array('count').getInfo()

    return pd.DataFrame({
        'doy'         : doys,
        'n_detections': [int(c) if c is not None else 0 for c in counts]
    })

In [7]:
# =============================================================================
# FUNCTION: compute_timing_metrics
# =============================================================================

def compute_timing_metrics(df, year):
    """
    Compute fire season timing metrics from a daily detection count DataFrame.

    Onset and end are defined by percentile thresholds on the cumulative
    detection fraction (5% and 95% respectively). Peak is defined as the
    fire activity centroid — the detection-weighted mean DOY — which is
    more robust to sparse outlier detections than the rolling-mean maximum
    and is guaranteed to fall within the onset–end window.

    Returns None if total detections fall below MIN_DETECTIONS, or if
    onset/end cannot be computed.

    Parameters
    ----------
    df : pd.DataFrame
        Daily counts DataFrame with columns 'doy' and 'n_detections',
        as returned by get_daily_counts().
    year : int
        The calendar year being processed (used for logging only).

    Returns
    -------
    dict or None
        Dict with keys: year, onset_doy, peak_doy, end_doy,
        season_length, n_detections.
        Returns None if metrics cannot be computed.
    """

    total = df['n_detections'].sum()

    if total < MIN_DETECTIONS:
        print(f'  {year}: insufficient detections ({total}), skipping.')
        return None

    df = df.copy().sort_values('doy')
    cumulative = df['n_detections'].cumsum()
    cum_frac   = cumulative / total

    # Onset: first DOY where cumulative fraction reaches 5%
    onset_rows = df[cum_frac >= ONSET_THRESHOLD]
    # End: first DOY where cumulative fraction reaches 95%
    end_rows   = df[cum_frac >= END_THRESHOLD]

    if onset_rows.empty or end_rows.empty:
        print(f'  {year}: could not compute onset or end, skipping.')
        return None

    onset_doy = int(onset_rows.iloc[0]['doy'])
    end_doy   = int(end_rows.iloc[0]['doy'])

    # Peak: fire activity centroid (detection-weighted mean DOY)
    # More robust than rolling-mean maximum; always falls within onset–end window
    weights   = df['n_detections']
    peak_doy  = int(round((df['doy'] * weights).sum() / weights.sum()))

    # Validate peak is within onset–end window
    if not (onset_doy <= peak_doy <= end_doy):
        print(f'  {year}: WARNING — peak ({peak_doy}) outside onset–end window '
              f'({onset_doy}–{end_doy}), flagging.')

    # Season length inclusive of both endpoints
    season_length = end_doy - onset_doy + 1

    return {
        'year'         : year,
        'onset_doy'    : onset_doy,
        'peak_doy'     : peak_doy,
        'end_doy'      : end_doy,
        'season_length': season_length,
        'n_detections' : int(total)
    }

In [8]:
# =============================================================================
# FULL PIPELINE LOOP — ALL ECOREGIONS x ALL YEARS
# =============================================================================
# Saves a per-ecoregion CSV after each ecoregion completes so progress
# is not lost if the run is interrupted.

output_dir = 'outputs/turkey_ecoregions'
os.makedirs(output_dir, exist_ok=True)

all_metrics = []

for eco in eco_records:
    eco_id   = eco['eco_id']
    eco_name = eco['eco_name']
    geometry = eco['geometry']

    print(f'\n=== {eco_name} (ID: {eco_id}) ===')
    eco_metrics = []

    for year in YEARS:
        t0 = time.time()

        try:
            df_year = get_daily_counts(geometry, year)
            metrics = compute_timing_metrics(df_year, year)

            if metrics is not None:
                metrics['eco_id']   = eco_id
                metrics['eco_name'] = eco_name
                eco_metrics.append(metrics)
                all_metrics.append(metrics)

        except Exception as e:
            print(f'  {year}: ERROR — {e}')
            continue

        t1 = time.time()
        print(f'  {year}: done in {t1 - t0:.1f}s')

    # Save per-ecoregion CSV immediately after finishing all years
    if eco_metrics:
        eco_df    = pd.DataFrame(eco_metrics)
        safe_name = eco_name.replace(' ', '_').replace('/', '_')
        eco_path  = f'{output_dir}/{eco_id}_{safe_name}.csv'
        eco_df.to_csv(eco_path, index=False)
        print(f'  Saved {len(eco_metrics)} years → {os.path.abspath(eco_path)}')
    else:
        print(f'  No valid years for {eco_name}, skipping CSV.')

print('\nAll ecoregions complete.')


=== Anatolian conifer and deciduous mixed forests (ID: 786) ===
  2003: done in 7.8s
  2004: done in 8.2s
  2005: done in 9.9s
  2006: done in 10.6s
  2007: done in 10.9s
  2008: done in 8.4s
  2009: done in 7.8s
  2010: done in 7.8s
  2011: done in 6.7s
  2012: done in 9.7s
  2013: done in 9.4s
  2014: done in 17.9s
  2015: done in 13.9s
  2016: done in 11.7s
  2017: done in 8.2s
  2018: done in 7.4s
  2019: done in 12.8s
  2020: done in 17.2s
  2021: done in 7.8s
  2022: done in 7.0s
  2023: done in 6.5s
  2024: done in 9.3s
  Saved 22 years → c:\Users\ibekar\Documents\GitProjects\TGPF\outputs\turkey_ecoregions\786_Anatolian_conifer_and_deciduous_mixed_forests.csv

=== Southern Anatolian montane conifer and deciduous forests (ID: 804) ===
  2003: done in 13.8s
  2004: done in 10.4s
  2005: done in 11.8s
  2006: done in 14.0s
  2007: done in 16.0s
  2008: done in 13.3s
  2009: done in 10.1s
  2010: done in 10.2s
  2011: done in 13.4s
  2012: done in 13.7s
  2013: done in 10.3s
  2014

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# COMBINE ALL RESULTS INTO MASTER CSV
# =============================================================================

master_df = pd.DataFrame(all_metrics)

# Reorder columns
master_df = master_df[['eco_id', 'eco_name', 'year',
                        'onset_doy', 'peak_doy', 'end_doy',
                        'season_length', 'n_detections']]

master_path = f'{output_dir}/master_turkey.csv'
master_df.to_csv(master_path, index=False)

print(f'Master CSV saved: {master_df.shape[0]} ecoregion-year rows.')
print(f'Path: {os.path.abspath(master_path)}')
print()
print(master_df.head(10))

In [ ]:
# =============================================================================
# PLOT — ONSET DOY PER ECOREGION ACROSS YEARS
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Fire Season Timing — Turkey Ecoregions (2003–2024)', fontsize=13)

metrics_to_plot = [
    ('onset_doy',     'Onset DOY',            'orange'),
    ('peak_doy',      'Peak DOY (centroid)',   'firebrick'),
    ('end_doy',       'End DOY',               'steelblue'),
    ('season_length', 'Season Length (days)',  'green'),
]

for ax, (metric, title, color) in zip(axes.flatten(), metrics_to_plot):
    for eco_name, group in master_df.groupby('eco_name'):
        ax.plot(group['year'], group[metric],
                marker='o', linewidth=1.5, label=eco_name)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_xticks(YEARS)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

# Single shared legend outside the plots
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center',
           ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.show()